<a href="https://www.kaggle.com/code/bwdelbnhawy/predicting-student-health-risk?scriptVersionId=338872271" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from category_encoders import CatBoostEncoder
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    balanced_accuracy_score, 
    classification_report, 
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


SEED = 42
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
print("🚀 Cell 1 Complete: Environment & Seed Locked!")

🚀 Cell 1 Complete: Environment & Seed Locked!


In [2]:

path_tr = "/kaggle/input/competitions/playground-series-s6e7/train.csv"
path_te = "/kaggle/input/competitions/playground-series-s6e7/test.csv"

train_df = pd.read_csv(path_tr, index_col="id")
test_df = pd.read_csv(path_te, index_col="id")
print(f"📊 Train: {train_df.shape} | Test: {test_df.shape}")

📊 Train: (690088, 14) | Test: (295753, 13)


In [3]:
train_df.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [4]:
train_df.describe()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,614089.000000,682255.000000,676190.000000,637235.000000,676172.000000,683187.000000,646611.000000
mean,6.992597,75.096504,22.984925,2226.084931,8615.953050,38.751456,2.188542
std,1.215407,8.175106,2.481787,347.532098,3929.399831,14.742189,0.518489
min,3.000000,50.000000,16.000000,1200.000000,1002.000000,0.000000,0.500000
25%,6.160000,69.400000,21.320000,2053.000000,5389.000000,29.200000,1.840000
50%,6.990000,75.100000,22.990000,2241.000000,8856.000000,39.400000,2.170000
75%,7.810000,80.700000,24.660000,2456.000000,12114.000000,49.400000,2.500000
max,10.000000,107.700000,34.820000,3580.000000,14999.000000,99.800000,4.720000


In [5]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           614089 non-null  float64
 2   heart_rate               682255 non-null  float64
 3   bmi                      676190 non-null  float64
 4   calorie_expenditure      637235 non-null  float64
 5   step_count               676172 non-null  float64
 6   exercise_duration        683187 non-null  float64
 7   water_intake             646611 non-null  float64
 8   diet_type                683187 non-null  object 
 9   stress_level             607277 non-null  object 
 10  sleep_quality            631757 non-null  object 
 11  physical_activity_level  653467 non-null  object 
 12  smoking_alcohol          661506 non-null  object 
 13  gender                   668715 non-null  object 
dtypes: float6

In [6]:
train_df.duplicated().sum()

np.int64(0)

In [ ]:
plt.figure(figsize=(8, 4))
ax = sns.countplot(data=train_df, x="health_condition", palette="viridis", order=train_df["health_condition"].value_counts().index)
plt.title("🎯 Target Class Distribution (Health Condition)", fontsize=13, fontweight="bold")
plt.xlabel("Health Condition")
plt.ylabel("Count")

for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 6), textcoords='offset points')

plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
missing_df = pd.DataFrame({
    "Missing_Count": train_df.isnull().sum(),
    "Missing_Percent": (train_df.isnull().sum() / len(train_df)) * 100
})
missing_df = missing_df[missing_df["Missing_Count"] > 0].sort_values("Missing_Percent", ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(data=missing_df, x="Missing_Percent", y=missing_df.index, palette="mako")
plt.title("📊 Percentage of Missing Values per Feature (%)", fontsize=13, fontweight="bold")
plt.xlabel("Missing Percentage (%)")
plt.grid(axis="x", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
temp_tr = train_df.copy()
temp_tr["target_encoded"] = LabelEncoder().fit_transform(temp_tr["health_condition"])
num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = temp_tr[num_cols + ["target_encoded"]].corr()

sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("🔥 Numerical Correlation Matrix (Features vs Target)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    sns.histplot(train_df[col], kde=True, ax=axes[0], color="#2b5c8f", bins=30)
    axes[0].set_title(f"📈 {col} Distribution", fontsize=11, fontweight="bold")
    
    sns.boxplot(x=train_df["health_condition"], y=train_df[col], ax=axes[1], palette="Set2")
    axes[1].set_title(f"📦 {col} across Health Conditions", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
cat_cols = train_df.select_dtypes(include=["object"]).columns.drop("health_condition").tolist()
for col in cat_cols:
    plt.figure(figsize=(8, 3.5))
    sns.countplot(data=train_df, x=col, hue="health_condition", palette="Set2")
    plt.title(f"📋 {col} vs Health Condition", fontsize=11, fontweight="bold")
    plt.xticks(rotation=15)
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [7]:

orig_cols = [c for c in train_df.columns if c != "health_condition"]
train_df["missing_count"] = train_df[orig_cols].isna().sum(axis=1)
test_df["missing_count"] = test_df[orig_cols].isna().sum(axis=1)

base_num_cols = ["bmi", "water_intake", "sleep_duration", "heart_rate", "exercise_duration", "step_count", "calorie_expenditure"]
for col in base_num_cols:
    if col in train_df.columns:
        train_df[f"{col}_null"] = train_df[col].isna().astype(np.int8)
        test_df[f"{col}_null"] = test_df[col].isna().astype(np.int8)

sleep_map = {"poor": 1, "average": 2, "good": 3}
stress_map = {"low": 1, "medium": 2, "high": 3}

for df in [train_df, test_df]:
    df["gender_diet"] = df["gender"].astype(str) + "_" + df["diet_type"].astype(str)
    df["activity_smoking"] = df["physical_activity_level"].astype(str) + "_" + df["smoking_alcohol"].astype(str)
    
    sq = df["sleep_quality"].map(sleep_map)
    st = df["stress_level"].map(stress_map)

    df["effective_sleep"] = df["sleep_duration"] * sq
    df["stress_sleep_ratio"] = np.where(df["sleep_duration"] > 0, st / df["sleep_duration"], np.nan)
    df["sleep_per_stress"] = np.where(st > 0, df["sleep_duration"] / st, np.nan)
    df["cardio_load"] = df["heart_rate"] * df["exercise_duration"]
    df["calories_per_1000_steps"] = np.where(df["step_count"] > 0, df["calorie_expenditure"] / (df["step_count"] / 1000.0), np.nan)
    df["water_per_bmi"] = np.where(df["bmi"] > 0, df["water_intake"] / df["bmi"], np.nan)

nominal_cols = ["gender", "diet_type", "physical_activity_level", "smoking_alcohol", "gender_diet", "activity_smoking", "sleep_quality", "stress_level"]

for c in nominal_cols:
    train_df[c] = train_df[c].fillna("Missing")
    test_df[c] = test_df[c].fillna("Missing")

feature_cols = [c for c in train_df.columns if c != "health_condition"]

label_enc = LabelEncoder()
y = label_enc.fit_transform(train_df["health_condition"])
X = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
n_classes = len(label_enc.classes_)
print("✅ Cell 3 Complete: Feature Engineering (Clean Redundancy Removal)!")

✅ Cell 3 Complete: Feature Engineering (Clean Redundancy Removal)!


In [ ]:
oof_lgb = np.zeros((len(train_df), n_classes))
test_lgb = np.zeros((len(test_df), n_classes))

print("🚀 Training Regularized LightGBM...")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[trn_idx].copy(), y[trn_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]
    X_te = X_test.copy()
    
    cb_enc = CatBoostEncoder(cols=nominal_cols, a=5, random_state=SEED, return_df=True)
    X_tr_enc = cb_enc.fit_transform(X_tr, y_tr)
    X_va_enc = cb_enc.transform(X_va)
    X_te_enc = cb_enc.transform(X_te)
    
    model_lgb = lgb.LGBMClassifier(
        n_estimators=1500, learning_rate=0.015, num_leaves=31, max_depth=7,
        min_child_samples=30, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(
        X_tr_enc, y_tr, eval_set=[(X_va_enc, y_va)], 
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va_enc)
    test_lgb += model_lgb.predict_proba(X_te_enc) / 5.0

lgb_acc = balanced_accuracy_score(y, np.argmax(oof_lgb, axis=1))
print(f"🔥 LightGBM Balanced Accuracy: {lgb_acc:.5f}")

In [8]:
import optuna
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight


sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)


def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            1500,
            4000,
            step=250
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.05,
            log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            4,
            8
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            5,
            30
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.7,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.7,
            1.0
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            1
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1,
            10
        )
    }


    model = xgb.XGBClassifier(
        **params,
        objective="multi:softprob",
        num_class=len(np.unique(y)),
        tree_method="hist",
        device="cuda",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )


    model.fit(
        X_train_enc,
        y_train,
        sample_weight=sample_weights,
        eval_set=[
            (X_val_enc, y_val)
        ],
        verbose=False
    )


    pred = model.predict(
        X_val_enc
    )


    score = balanced_accuracy_score(
        y_val,
        pred
    )


    return score



study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=42
    )
)


study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True
)


best_params = study.best_params


best_score = study.best_value


best_params, best_score

NameError: name 'y_train' is not defined

In [ ]:
oof_xgb = np.zeros((len(train_df), n_classes))
test_xgb = np.zeros((len(test_df), n_classes))
xgb_importance = None

print("🎯 Training XGBoost GPU Model...")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[trn_idx].copy(), y[trn_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]
    X_te = X_test.copy()

    cb_enc = CatBoostEncoder(cols=nominal_cols, a=5, random_state=SEED, return_df=True)
    X_tr_enc = cb_enc.fit_transform(X_tr, y_tr)
    X_va_enc = cb_enc.transform(X_va)
    X_te_enc = cb_enc.transform(X_te)
    
    sample_weights = compute_sample_weight("balanced", y_tr)

    model_xgb = XGBClassifier(
        n_estimators=2500, learning_rate=0.0105, max_depth=5, min_child_weight=14,
        subsample=0.85, colsample_bytree=0.87, tree_method="hist", device="cuda",
        early_stopping_rounds=50, random_state=SEED
    )
    model_xgb.fit(X_tr_enc, y_tr, sample_weight=sample_weights, eval_set=[(X_va_enc, y_va)], verbose=False)
    
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va_enc)
    test_xgb += model_xgb.predict_proba(X_te_enc) / 5.0
    
    if xgb_importance is None:
        xgb_importance = np.zeros(len(model_xgb.feature_importances_))
    xgb_importance += model_xgb.feature_importances_ / 5.0

xgb_acc = balanced_accuracy_score(y, np.argmax(oof_xgb, axis=1))
print(f"🎯 XGBoost Balanced Accuracy: {xgb_acc:.5f}")

In [ ]:
oof_cat = np.zeros((len(train_df), n_classes))
test_cat = np.zeros((len(test_df), n_classes))

print("🔮 Training CatBoost GPU Model...")
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[trn_idx].copy(), y[trn_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]
    X_te = X_test.copy()

    cat_cols_idx = [X_tr.columns.get_loc(c) for c in nominal_cols]
    X_tr_cat, X_va_cat, X_te_cat = X_tr.copy(), X_va.copy(), X_te.copy()
    for c in nominal_cols:
        X_tr_cat[c] = X_tr_cat[c].astype(str)
        X_va_cat[c] = X_va_cat[c].astype(str)
        X_te_cat[c] = X_te_cat[c].astype(str)

    model_cat = CatBoostClassifier(
        iterations=2500, learning_rate=0.04, depth=7, auto_class_weights="Balanced",
        task_type="GPU", random_seed=SEED, verbose=0
    )
    model_cat.fit(X_tr_cat, y_tr, cat_features=cat_cols_idx, eval_set=(X_va_cat, y_va), early_stopping_rounds=50)
    
    oof_cat[val_idx] = model_cat.predict_proba(X_va_cat)
    test_cat += model_cat.predict_proba(X_te_cat) / 5.0

cat_acc = balanced_accuracy_score(y, np.argmax(oof_cat, axis=1))
print(f"🔮 CatBoost Balanced Accuracy: {cat_acc:.5f}")

In [ ]:
print("🤖 Training Stacking Meta-Model...")
X_stack = np.hstack([oof_xgb, oof_lgb, oof_cat])
X_test_stack = np.hstack([test_xgb, test_lgb, test_cat])

oof_stack = np.zeros_like(oof_xgb)
test_stack = np.zeros_like(test_xgb)

for trn_idx, val_idx in skf.split(X_stack, y):
    meta = LogisticRegression(C=0.1, max_iter=1000, class_weight="balanced", random_state=SEED)
    meta.fit(X_stack[trn_idx], y[trn_idx])
    
    oof_stack[val_idx] = meta.predict_proba(X_stack[val_idx])
    test_stack += meta.predict_proba(X_test_stack) / 5.0

acc_stack = balanced_accuracy_score(y, np.argmax(oof_stack, axis=1))


oof_ens = (oof_xgb + oof_lgb + oof_cat) / 3.0
test_ens = (test_xgb + test_lgb + test_cat) / 3.0
acc_ens = balanced_accuracy_score(y, np.argmax(oof_ens, axis=1))

models_dict = {
    "XGBoost": (xgb_acc, oof_xgb, test_xgb),
    "LightGBM": (lgb_acc, oof_lgb, test_lgb),
    "CatBoost": (cat_acc, oof_cat, test_cat),
    "Blended Ensemble": (acc_ens, oof_ens, test_ens),
    "FoldAligned Stacking": (acc_stack, oof_stack, test_stack)
}

best_model_name = max(models_dict, key=lambda k: models_dict[k][0])
best_acc, best_oof_probs, best_test_probs = models_dict[best_model_name]
best_preds = np.argmax(best_oof_probs, axis=1)

print("\n🏆 ===== ALL STRATEGIES BENCHMARK =====")
for name, (score, _, _) in models_dict.items():
    print(f"🔹 {name:<20}: Balanced Acc = {score:.5f}")

print(f"\n🥇 Absolute Champion Strategy Selected: 【 {best_model_name} 】 with Score = {best_acc:.5f}\n")

In [ ]:
print(f"📄 ===== {best_model_name} CLASSIFICATION REPORT =====")
print(classification_report(y, best_preds, target_names=label_enc.classes_))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y, best_preds), annot=True, fmt="d", cmap="Blues",
            xticklabels=label_enc.classes_, yticklabels=label_enc.classes_)
plt.title(f"🧩 {best_model_name} - Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


fi_df = pd.DataFrame({
    'Feature': feature_cols, 
    'Importance': model_lgb.feature_importances_
}).sort_values(by='Importance', ascending=False)


plt.figure(figsize=(10, 6))
sns.barplot(
    data=fi_df, 
    x='Importance', 
    y='Feature', 
    palette='viridis'
)

plt.title('Top Feature Importances (Champion Model: LightGBM)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score (Split Count)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


print("=== Top Features Table ===")
print(fi_df.reset_index(drop=True))

In [ ]:
y_bin = label_binarize(y, classes=range(n_classes))

plt.figure(figsize=(7, 4))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], best_oof_probs[:, i])
    plt.plot(fpr, tpr, lw=2, label=f"Class {label_enc.classes_[i]} (AUC = {auc(fpr, tpr):.4f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
plt.title(f"📈 {best_model_name} - ROC-AUC Curves")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
for i in range(n_classes):
    precision, recall, _ = precision_recall_curve(y_bin[:, i], best_oof_probs[:, i])
    plt.plot(recall, precision, lw=2, label=f"Class {label_enc.classes_[i]} (AP = {average_precision_score(y_bin[:, i], best_oof_probs[:, i]):.4f})")
plt.title(f"📉 {best_model_name} - Precision-Recall (PR) Curves")
plt.legend(loc="lower left")
plt.tight_layout()
plt.show()

In [ ]:
print("⏳ Benchmarking All Classic Machine Learning Algorithms (Strict Identical 5-Fold)...")

sample_size = min(100000, len(X))
sample_idx = np.random.choice(len(X), size=sample_size, replace=False)
X_sub, y_sub = X.iloc[sample_idx].copy(), y[sample_idx]

all_models = {
    "Logistic Regression": Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=300, class_weight='balanced'))]),
    "Decision Tree": Pipeline([('imputer', SimpleImputer()), ('clf', DecisionTreeClassifier(max_depth=10, class_weight='balanced'))]),
    "Random Forest": Pipeline([('imputer', SimpleImputer()), ('clf', RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', n_jobs=-1))]),
    "Extra Trees": Pipeline([('imputer', SimpleImputer()), ('clf', ExtraTreesClassifier(n_estimators=100, max_depth=12, class_weight='balanced', n_jobs=-1))]),
    "AdaBoost": Pipeline([('imputer', SimpleImputer()), ('clf', AdaBoostClassifier(n_estimators=50))]),
    "Gaussian Naive Bayes": Pipeline([('imputer', SimpleImputer()), ('clf', GaussianNB())]),
    "K-Nearest Neighbors": Pipeline([('imputer', SimpleImputer()), ('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))])
}

benchmark_scores = {}

for name, clf in all_models.items():
    scores = []
    for trn_i, val_i in skf.split(X_sub, y_sub):
        X_tr_b, y_tr_b = X_sub.iloc[trn_i].copy(), y_sub[trn_i]
        X_va_b, y_va_b = X_sub.iloc[val_i].copy(), y_sub[val_i]

        cb_b = CatBoostEncoder(cols=nominal_cols, a=5, random_state=SEED, return_df=True)
        X_tr_b_enc = cb_b.fit_transform(X_tr_b, y_tr_b)
        X_va_b_enc = cb_b.transform(X_va_b)
        
        clf.fit(X_tr_b_enc, y_tr_b)
        preds = clf.predict(X_va_b_enc)
        scores.append(balanced_accuracy_score(y_va_b, preds))
    benchmark_scores[name] = np.mean(scores)

benchmark_scores["LightGBM (GBDT)"] = lgb_acc
benchmark_scores["CatBoost (GBDT)"] = cat_acc
benchmark_scores["XGBoost (GBDT)"] = xgb_acc

df_benchmark = pd.DataFrame(list(benchmark_scores.items()), columns=["Algorithm", "Balanced Accuracy"]).sort_values("Balanced Accuracy", ascending=False)
display(df_benchmark)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_benchmark, x="Balanced Accuracy", y="Algorithm", palette="crest")
plt.title("📊 All Classification Algorithms Benchmark Comparison (Strict Identical 5-Fold)")
plt.tight_layout()
plt.show()

In [ ]:
final_labels = label_enc.inverse_transform(np.argmax(best_test_probs, axis=1))
sub = pd.DataFrame({"id": test_df.index, "health_condition": final_labels})
sub.to_csv("submission.csv", index=False)

print(f"\n🎉 'submission.csv' Created Successfully using 【 {best_model_name} 】!")

In [ ]:
import joblib

print("💾 Saving the trained LightGBM model directly from memory...")


joblib.dump(model_lgb, "lgb_health_model.pkl")


joblib.dump(cb_enc, "catboost_encoder.pkl")
joblib.dump(label_enc, "label_encoder.pkl")

print("🎉 Saved instantly from current run! No extra training needed.")